# Glass Key 112-Byte Notebook

This notebook gives you a **real runnable workspace** for the Glass Key architecture.

It supports two workflows:

1. **Real file mode**  
   Choose any file on disk, inspect it, hash it, estimate entropy, and test whether it can be represented by the current Glass Key model.

2. **Model-defined stream mode**  
   Generate an **exact 112-byte Glass Key** for a large virtual file produced by a deterministic generator:
   - **48-byte seed**
   - **64-byte SHA-512 anchor**
   - exact reconstruction possible with the same decoder

## Hard boundary

A fixed 112-byte key **cannot** losslessly encode an arbitrary file in general.

What *is* possible is exact reconstruction for a **restricted source class**:
files already defined by the generator model.

That is what this notebook demonstrates clearly and honestly.


In [9]:
# Optional install cell
# Uncomment only if your notebook environment is missing something.
# %pip install ipywidgets

print("Setup cell ready.")


Setup cell ready.


In [10]:
from __future__ import annotations

import hashlib
import math
import struct
import json
from dataclasses import dataclass
from pathlib import Path
from collections import Counter

# =========================
# USER CONFIG
# =========================

# Choose a real file here when using REAL_FILE mode.
FILE_PATH = Path(r"d://nexus//glasskey_notebook_out/Hurt.flac")   # e.g. Path(r"C:\\data\\myfile.bin")

# Output folder for generated artifacts.
OUT_DIR = Path("d://nexus//glasskey_notebook_out")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Modes:
#   "real_file"     -> inspect any chosen file honestly
#   "model_virtual" -> create an exact 112-byte Glass Key for a generator-defined file
MODE = "real_file"

# If MODE == "model_virtual":
# If VIRTUAL_LENGTH is None and FILE_PATH exists, the chosen file's size is reused
# for the virtual stream length.
VIRTUAL_LENGTH = None
USER_SEED_HEX = "00112233445566778899aabbccddeeff"   # must be 16 bytes = 32 hex chars
NONCE = 42
CHUNK_SIZE = 1 << 20  # 1 MiB

# For previews and entropy estimation:
PREVIEW_BYTES = 1 << 20
ENTROPY_SAMPLE_BYTES = 1 << 20

print("MODE =", MODE)
print("FILE_PATH =", FILE_PATH)
print("OUT_DIR =", OUT_DIR.resolve())


MODE = real_file
FILE_PATH = d:\nexus\glasskey_notebook_out\Hurt.flac
OUT_DIR = D:\Nexus\glasskey_notebook_out


In [11]:
# =========================
# CORE HELPERS
# =========================

MAGIC = b"GK12"
VERSION = 1
MODEL_SHAKE_CHUNK = 1
SEED_SIZE = 48
ANCHOR_SIZE = 64
KEY_SIZE = 112

@dataclass(frozen=True)
class Seed48:
    magic: bytes
    version: int
    model_id: int
    flags: int
    length: int
    chunk_size: int
    nonce: int
    user_seed: bytes

    def pack(self) -> bytes:
        if len(self.magic) != 4:
            raise ValueError("magic must be 4 bytes")
        if len(self.user_seed) != 16:
            raise ValueError("user_seed must be 16 bytes")
        return (
            self.magic
            + struct.pack(
                ">BBHQQQ",
                self.version,
                self.model_id,
                self.flags,
                self.length,
                self.chunk_size,
                self.nonce,
            )
            + self.user_seed
        )

    @staticmethod
    def unpack(blob: bytes) -> "Seed48":
        if len(blob) != SEED_SIZE:
            raise ValueError(f"Seed must be exactly {SEED_SIZE} bytes")
        magic = blob[:4]
        version, model_id, flags, length, chunk_size, nonce = struct.unpack(
            ">BBHQQQ", blob[4:32]
        )
        user_seed = blob[32:48]
        return Seed48(magic, version, model_id, flags, length, chunk_size, nonce, user_seed)

def human_bytes(n: int) -> str:
    units = ["B", "KiB", "MiB", "GiB", "TiB"]
    x = float(n)
    idx = 0
    while x >= 1024 and idx < len(units) - 1:
        x /= 1024
        idx += 1
    return f"{x:.2f} {units[idx]}"

def sha256_file(path: Path, block_size: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            chunk = f.read(block_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def sha512_file(path: Path, block_size: int = 1 << 20) -> str:
    h = hashlib.sha512()
    with path.open("rb") as f:
        while True:
            chunk = f.read(block_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def estimate_shannon_entropy(path: Path, sample_bytes: int = 1 << 20) -> float:
    with path.open("rb") as f:
        data = f.read(sample_bytes)
    if not data:
        return 0.0
    counts = Counter(data)
    total = len(data)
    entropy = 0.0
    for c in counts.values():
        p = c / total
        entropy -= p * math.log2(p)
    return entropy

def make_seed(length: int, user_seed: bytes, nonce: int = 0, chunk_size: int = 1 << 20) -> bytes:
    seed = Seed48(
        magic=MAGIC,
        version=VERSION,
        model_id=MODEL_SHAKE_CHUNK,
        flags=0,
        length=length,
        chunk_size=chunk_size,
        nonce=nonce,
        user_seed=user_seed,
    )
    return seed.pack()

def iter_generated_chunks(seed48: bytes, limit: int | None = None):
    seed = Seed48.unpack(seed48)
    total = seed.length if limit is None else min(seed.length, limit)
    emitted = 0
    index = 0
    while emitted < total:
        take = min(seed.chunk_size, total - emitted)
        h = hashlib.shake_256()
        h.update(seed48)
        h.update(index.to_bytes(8, "big"))
        yield h.digest(take)
        emitted += take
        index += 1

def compute_anchor(seed48: bytes) -> bytes:
    h = hashlib.sha512()
    for chunk in iter_generated_chunks(seed48):
        h.update(chunk)
    return h.digest()

def make_key112(length: int, user_seed: bytes, nonce: int = 0, chunk_size: int = 1 << 20) -> bytes:
    seed48 = make_seed(length, user_seed, nonce, chunk_size)
    anchor64 = compute_anchor(seed48)
    return seed48 + anchor64

def parse_key112(key112: bytes):
    if len(key112) != KEY_SIZE:
        raise ValueError(f"Key must be exactly {KEY_SIZE} bytes")
    seed48 = key112[:SEED_SIZE]
    anchor64 = key112[SEED_SIZE:]
    return Seed48.unpack(seed48), anchor64

def verify_key112(key112: bytes) -> bool:
    seed, anchor = parse_key112(key112)
    expected = compute_anchor(seed.pack())
    return expected == anchor

def materialize_key112(key112: bytes, out_path: Path, limit: int | None = None) -> int:
    seed, _ = parse_key112(key112)
    written = 0
    with out_path.open("wb") as f:
        for chunk in iter_generated_chunks(seed.pack(), limit=limit):
            f.write(chunk)
            written += len(chunk)
    return written

print("Helpers loaded.")


Helpers loaded.


In [12]:
# =========================
# REAL FILE MODE
# =========================

if MODE == "real_file":
    if not FILE_PATH or not FILE_PATH.exists():
        raise FileNotFoundError("Set FILE_PATH to an existing file first.")

    size = FILE_PATH.stat().st_size
    info = {
        "file": str(FILE_PATH),
        "size_bytes": size,
        "size_human": human_bytes(size),
        "sha256": sha256_file(FILE_PATH),
        "sha512": sha512_file(FILE_PATH),
        "estimated_entropy_bits_per_byte": estimate_shannon_entropy(FILE_PATH, ENTROPY_SAMPLE_BYTES),
        "fixed_112_byte_exact_recovery_possible_for_arbitrary_file": False,
        "reason": (
            "A fixed 112-byte payload cannot losslessly identify every possible file of this size. "
            "The Glass Key architecture only gives exact reconstruction for generator-defined source classes."
        ),
    }

    print(json.dumps(info, indent=2))

    report_path = OUT_DIR / (FILE_PATH.name + ".glasskey_report.json")
    report_path.write_text(json.dumps(info, indent=2), encoding="utf-8")
    print(f"Saved report to: {report_path.resolve()}")

    anchor_path = OUT_DIR / (FILE_PATH.name + ".sha512.txt")
    anchor_path.write_text(info["sha512"], encoding="utf-8")
    print(f"Saved SHA-512 anchor to: {anchor_path.resolve()}")
else:
    print("Skipping REAL_FILE mode.")


{
  "file": "d:\\nexus\\glasskey_notebook_out\\Hurt.flac",
  "size_bytes": 35685979,
  "size_human": "34.03 MiB",
  "sha256": "1d2e55dbf1ba547ad77646760789ca8f44bb1df6da8b2db9ee2b9251247be34b",
  "sha512": "eef257591cb1d83ef08db464041bd28522c334650b1e0fc53d09b1fdf6ef0f50dafda06912010b7b3b6ac6b8383e16258650f20226c2d8b3180bf64bc8bb7c95",
  "estimated_entropy_bits_per_byte": 7.97198298644883,
  "fixed_112_byte_exact_recovery_possible_for_arbitrary_file": false,
  "reason": "A fixed 112-byte payload cannot losslessly identify every possible file of this size. The Glass Key architecture only gives exact reconstruction for generator-defined source classes."
}
Saved report to: D:\Nexus\glasskey_notebook_out\Hurt.flac.glasskey_report.json
Saved SHA-512 anchor to: D:\Nexus\glasskey_notebook_out\Hurt.flac.sha512.txt


In [13]:
# =========================
# MODEL-DEFINED 112-BYTE GLASS KEY MODE
# =========================

if MODE == "model_virtual":
    if VIRTUAL_LENGTH is None:
        if FILE_PATH and FILE_PATH.exists():
            length = FILE_PATH.stat().st_size
        else:
            raise ValueError("Set VIRTUAL_LENGTH, or point FILE_PATH at an existing file to reuse its size.")
    else:
        length = int(VIRTUAL_LENGTH)

    user_seed = bytes.fromhex(USER_SEED_HEX)
    if len(user_seed) != 16:
        raise ValueError("USER_SEED_HEX must decode to exactly 16 bytes")

    key112 = make_key112(length=length, user_seed=user_seed, nonce=NONCE, chunk_size=CHUNK_SIZE)
    seed, anchor = parse_key112(key112)

    key_path = OUT_DIR / f"virtual_{length}_bytes.gk"
    key_path.write_bytes(key112)

    preview_path = OUT_DIR / f"virtual_{length}_bytes.preview.bin"
    preview_written = materialize_key112(key112, preview_path, limit=PREVIEW_BYTES)

    info = {
        "virtual_length_bytes": seed.length,
        "virtual_length_human": human_bytes(seed.length),
        "seed_size_bytes": SEED_SIZE,
        "anchor_size_bytes": ANCHOR_SIZE,
        "total_key_size_bytes": KEY_SIZE,
        "compression_ratio": seed.length / KEY_SIZE,
        "seed48_hex": key112[:48].hex(),
        "anchor64_hex": anchor.hex(),
        "verify": verify_key112(key112),
        "key_path": str(key_path.resolve()),
        "preview_path": str(preview_path.resolve()),
        "preview_written_bytes": preview_written,
    }

    print(json.dumps(info, indent=2))
else:
    print("Skipping MODEL-DEFINED mode.")


Skipping MODEL-DEFINED mode.


In [14]:
# =========================
# OPTIONAL: MATERIALIZE FULL VIRTUAL FILE FROM A .gk KEY
# =========================

GK_PATH = None  # e.g. OUT_DIR / "virtual_1073741824_bytes.gk"

if GK_PATH:
    GK_PATH = Path(GK_PATH)
    key112 = GK_PATH.read_bytes()
    out_path = OUT_DIR / (GK_PATH.stem + ".full.bin")
    written = materialize_key112(key112, out_path)
    print(f"Materialized {written} bytes to {out_path.resolve()}")
else:
    print("Set GK_PATH if you want to materialize a full virtual file.")


Set GK_PATH if you want to materialize a full virtual file.


## How to use this

### To inspect a real file
1. Set:
   ```python
   MODE = "real_file"
   FILE_PATH = Path(r"...your file...")
   ```
2. Run all cells.
3. The notebook will:
   - hash the file
   - estimate entropy
   - save a JSON report
   - tell you honestly that arbitrary exact 112-byte recovery is not possible in general

### To generate a true 112-byte Glass Key
1. Set:
   ```python
   MODE = "model_virtual"
   VIRTUAL_LENGTH = ...   # or reuse FILE_PATH size
   USER_SEED_HEX = "00112233445566778899aabbccddeeff"
   NONCE = 42
   ```
2. Run all cells.
3. The notebook will:
   - generate a 112-byte `.gk`
   - verify it
   - write a preview binary
   - optionally materialize the full virtual file

## Why this notebook exists

This is the honest middle ground:

- it lets you choose **any real file**
- it shows the hard mathematical limit for arbitrary-file compression
- and it also demonstrates the exact **seed + anchor = large reconstructable object** architecture for model-defined data


In [15]:
import numpy as np
import hashlib
import struct

class GlassKeyCompressor:
    """
    Implements the 112-byte Glass Key compression protocol.
    Partitioned into:
    - 48-Byte Seed: Top 16 FFT bins (index, amplitude, phase) [5]
    - 64-Byte Anchor: SHA-256 hash + 32 bytes metadata [5, 9]
    """
    def __init__(self, harmonic_threshold=5.0):
        self.threshold = harmonic_threshold

    def compress(self, data):
        # 1. Calculate the Lossless Anchor (32-byte SHA-256) [5, 10]
        data_bytes = np.array(data, dtype=np.uint8).tobytes()
        data_hash = hashlib.sha256(data_bytes).digest()
        
        # 2. Extract Harmonic Seed (Top 16 FFT bins) [5, 11]
        # Treat data as a waveform to find its 'frozen verb' logic
        fft_vals = np.fft.rfft(data)
        amplitudes = np.abs(fft_vals)
        top_indices = np.argsort(amplitudes)[-16:] # Top 16 harmonics
        
        # 48-byte Seed generation (Index, Amp, Phase quantized to 1 byte each)
        seed = bytearray()
        for idx in top_indices:
            # Quantize values to 0-255 range for byte storage
            amp = int(np.clip(amplitudes[idx], 0, 255))
            phase = int(((np.angle(fft_vals[idx]) + np.pi) / (2 * np.pi)) * 255)
            seed.extend(struct.pack('BBB', idx % 256, amp, phase))
        
        # 3. Build 64-byte Anchor (Hash + Metadata) [4, 5]
        # Metadata includes original length and harmonic stats for unspooling
        metadata = struct.pack('>IIf', len(data), len(fft_vals), np.mean(amplitudes))
        metadata = metadata.ljust(32, b'\x00') # Pad metadata to 32 bytes
        
        anchor = data_hash + metadata
        return bytes(seed) + anchor

    def decompress(self, package):
        if len(package) != 112:
            raise ValueError("Invalid Glass Key footprint size.")
            
        # Separate Seed (48B) and Anchor (64B) [1, 5]
        seed = package[:48]
        data_hash = package[48:80]
        metadata = package[80:]
        
        orig_len, n_fft, _ = struct.unpack('>IIf', metadata[:12])
        
        # 4. Unspooling via Inverse Fast Fourier Transform (IFFT) [7, 11]
        # Reconstruct the frequency domain from the 16-state generator
        recon_fft = np.zeros(n_fft, dtype=complex)
        for i in range(16):
            idx, amp, ph_raw = struct.unpack('BBB', seed[i*3 : (i+1)*3])
            phase = (ph_raw / 255.0) * 2 * np.pi - np.pi
            recon_fft[idx] = amp * np.exp(1j * phase)
            
        # Unfold the 1D sequence back from the seed [8, 10]
        reconstructed_data = np.fft.irfft(recon_fft, n=orig_len)
        reconstructed_bytes = np.clip(reconstructed_data, 0, 255).astype(np.uint8).tobytes()
        
        # 5. Validation against the Anchor [4, 7]
        current_hash = hashlib.sha256(reconstructed_bytes).digest()
        if current_hash != data_hash:
            print("Warning: Arc-chord drift detected. Signal requires higher harmonic coherence.")
            
        return reconstructed_bytes